In [ ]:
# ============================================================
# EXERCISE 4 - AGE ESTIMATION GRAD-CAM
# Compare ResNet-50 vs Swin-Base
#
# Run AFTER the Exercise 4 training block.
# ============================================================

!pip -q install grad-cam

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


# ------------------------------------------------------------
# 1. COMBINE FROZEN BACKBONE + TRAINED AGE HEAD
# ------------------------------------------------------------

class AgeModel(nn.Module):

    def __init__(self, backbone, head):

        super().__init__()

        self.backbone = backbone
        self.head = head

    def forward(self, x):

        features = self.backbone(x)

        normalized_age = self.head(features)

        return normalized_age


resnet_age_model = AgeModel(
    resnet_backbone,
    resnet_head
).to(device)


swin_age_model = AgeModel(
    swin_backbone,
    swin_head
).to(device)


resnet_age_model.eval()
swin_age_model.eval()


# ------------------------------------------------------------
# 2. REGRESSION TARGET
# ------------------------------------------------------------

class RegressionTarget:

    def __call__(self, model_output):

        # One scalar output = normalized predicted age
        return model_output[0]


# ------------------------------------------------------------
# 3. SWIN RESHAPE
#
# torchvision Swin stores:
# B x H x W x C
#
# Grad-CAM needs:
# B x C x H x W
# ------------------------------------------------------------

def swin_reshape_transform(tensor):

    return tensor.permute(
        0, 3, 1, 2
    )


# ------------------------------------------------------------
# 4. GET CAM FOR ONE MODEL
# ------------------------------------------------------------

def get_age_cam(
    model,
    model_name,
    sample_index
):

    row = test_df.iloc[sample_index]

    filename = row["filenames"]
    true_age = float(row["age"])

    image_path = os.path.join(
        IMAGE_DIR,
        filename
    )

    original_image = Image.open(
        image_path
    ).convert("RGB")

    display_image = original_image.resize(
        (224, 224)
    )

    rgb_image = (
        np.array(display_image)
        .astype(np.float32)
        / 255.0
    )

    input_tensor = (
        transform(original_image)
        .unsqueeze(0)
        .to(device)
    )

    input_tensor.requires_grad_(True)

    # --------------------------------------------------------
    # AGE PREDICTION
    # --------------------------------------------------------

    with torch.no_grad():

        normalized_prediction = model(
            input_tensor
        ).item()

    predicted_age = (
        normalized_prediction
        * age_std.item()
        + age_mean.item()
    )


    # --------------------------------------------------------
    # GRAD-CAM TARGET LAYER
    # --------------------------------------------------------

    if model_name == "ResNet-50":

        target_layers = [
            model.backbone.layer4[-1]
        ]

        cam = GradCAM(
            model=model,
            target_layers=target_layers
        )

    else:

        target_layers = [
            model.backbone.features[-1][-1].norm1
        ]

        cam = GradCAM(
            model=model,
            target_layers=target_layers,
            reshape_transform=swin_reshape_transform
        )


    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=[RegressionTarget()]
    )[0]


    visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True
    )


    return (
        rgb_image,
        visualization,
        true_age,
        predicted_age
    )


# ------------------------------------------------------------
# 5. COMPARE BOTH MODELS ON SAME X-RAY
# ------------------------------------------------------------

def compare_age_models(sample_index):

    (
        image,
        resnet_cam,
        true_age,
        resnet_age
    ) = get_age_cam(
        resnet_age_model,
        "ResNet-50",
        sample_index
    )

    (
        _,
        swin_cam,
        _,
        swin_age
    ) = get_age_cam(
        swin_age_model,
        "Swin-Base",
        sample_index
    )


    plt.figure(
        figsize=(15, 5)
    )


    # Original
    plt.subplot(1, 3, 1)

    plt.imshow(image)

    plt.title(
        f"Original X-ray\n"
        f"Actual age: {true_age:.0f}"
    )

    plt.axis("off")


    # ResNet
    plt.subplot(1, 3, 2)

    plt.imshow(resnet_cam)

    plt.title(
        f"ResNet-50 Grad-CAM\n"
        f"Predicted age: {resnet_age:.1f}"
    )

    plt.axis("off")


    # Swin
    plt.subplot(1, 3, 3)

    plt.imshow(swin_cam)

    plt.title(
        f"Swin-Base Grad-CAM\n"
        f"Predicted age: {swin_age:.1f}"
    )

    plt.axis("off")


    plt.tight_layout()

    filename = f"age_CAM_{int(true_age)}.png"

    plt.savefig(
        os.path.join(SAVE_DIR, filename),
        dpi=300,
        bbox_inches="tight"
    )

    plt.tight_layout()

    filename = f"age_CAM_{int(true_age)}.png"

    plt.savefig(
        os.path.join(SAVE_DIR, filename),
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ------------------------------------------------------------
# 6. SELECT YOUNG, MIDDLE, OLD SUBJECTS
# ------------------------------------------------------------

young_index = test_df["age"].idxmin()

middle_index = (
    test_df["age"]
    - test_df["age"].median()
).abs().idxmin()

old_index = test_df["age"].idxmax()


# ------------------------------------------------------------
# 7. SHOW RESULTS
# ------------------------------------------------------------

print("YOUNG SUBJECT")
compare_age_models(
    young_index
)

print("MIDDLE-AGED SUBJECT")
compare_age_models(
    middle_index
)

print("OLDER SUBJECT")
compare_age_models(
    old_index
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

SAVE_DIR = "/content/drive/MyDrive/BME507/Warmup_3_4"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving everything to:", SAVE_DIR)